# PPO from scratch on CarRacing-v3

Task 6. Trains the implementation in `task6-ppo-from-scratch/ppo_carracing.py`.

**Read this before you start.** CarRacing is CPU-bound, not GPU-bound. Every step runs 8 Box2D
physics frames and renders 8 images, all on the CPU, while the tiny network waits on the GPU.
Free Colab gives you 2 vCPUs, and measured throughput here was 13.7 steps/sec with one actor,
which puts 1M steps at roughly 20 hours. A GPU does not fix this.

If you have a multi-core machine, run this locally instead and set `--num-envs` to your core
count. See the README. Use this notebook for a shorter run or if Colab is all you have.

Runtime -> Change runtime type -> T4 GPU before running anything.

## 1. Check the GPU

In [ ]:
!nvidia-smi
import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())

## 2. Decide where results go

Run this cell either way, it defines `OUT_DIR` which every later cell uses.

Set `USE_DRIVE = False` to keep everything in the temporary runtime, which is fine for a
short run. For anything over an hour leave it `True`. Colab disconnects idle notebooks and
the runtime disk goes with it.

In [ ]:
import os

USE_DRIVE = True   # False keeps results in the runtime instead

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/ppo_carracing_results'
else:
    OUT_DIR = '/content/results'

os.makedirs(OUT_DIR, exist_ok=True)
print('results will go to', OUT_DIR)

## 3. Install Box2D

SWIG has to be installed before Box2D, otherwise the build fails. Restart the runtime if Colab asks you to.

In [ ]:
!pip install -q swig
!pip install -q "gymnasium[box2d]" moviepy

import gymnasium as gym
env = gym.make('CarRacing-v3')
print('gymnasium', gym.__version__)
print('observation', env.observation_space.shape, '| action', env.action_space)
env.close()

## 4. Get the code

In [ ]:
%cd /content
!rm -rf aiea-internship
!git clone -q https://github.com/willb31/aiea-internship.git
%cd /content/aiea-internship/task6-ppo-from-scratch
!ls

## 5. Smoke test

Two thousand steps, just to confirm the loop runs and nothing is NaN. Should finish in a minute or two. Do not expect the car to do anything sensible yet.

In [ ]:
!python ppo_carracing.py --steps 4000 --run-name smoketest --out-dir /content/smoketest

## 6. The real run

`STEPS` is set to 200,000, which is about 4 hours on free Colab and enough to see the reward
curve start bending upward. Raise it only if you are prepared to chain sessions with the
resume cell below. The best checkpoint saves automatically whenever the 100-episode average
improves.

In [ ]:
RUN_NAME = 'run1'
STEPS = 200_000

!python ppo_carracing.py \
    --steps {STEPS} \
    --run-name {RUN_NAME} \
    --out-dir {OUT_DIR} \
    --num-envs 2 \
    --seed 0

### If it disconnected, run this to continue

Free Colab drops long sessions. Re-run cells 1, 2, 3 and 4 first (GPU check, OUT_DIR, install, clone),
then run this cell instead of cell 6. It picks up from the last checkpoint with the optimizer state
intact. Repeat as many times as needed. `--steps` is an absolute target, so leave it at your goal.

In [ ]:
CKPT = f'{OUT_DIR}/checkpoints/{RUN_NAME}_latest.pt'

import os
assert os.path.exists(CKPT), f'no checkpoint at {CKPT} yet, run cell 6 first'

!python ppo_carracing.py \
    --steps {STEPS} \
    --run-name {RUN_NAME} \
    --out-dir {OUT_DIR} \
    --resume {CKPT}

## 7. Graphs

In [ ]:
!python plot_results.py --run-name {RUN_NAME} --out-dir {OUT_DIR} --clip 0.1

from IPython.display import Image, display
import os
fig_dir = os.path.join(OUT_DIR, 'figures')
for name in sorted(os.listdir(fig_dir)):
    print(name)
    display(Image(os.path.join(fig_dir, name), width=900))

## 8. Watch it drive

Runs the best checkpoint with the distribution mean instead of a sample, so the driving is deterministic, and records video.

In [ ]:
!python ppo_carracing.py \
    --eval {OUT_DIR}/checkpoints/{RUN_NAME}_best.pt \
    --episodes 5 --video \
    --run-name {RUN_NAME} --out-dir {OUT_DIR}

In [ ]:
import glob, base64
from IPython.display import HTML

videos = sorted(glob.glob(os.path.join(OUT_DIR, 'videos', '*.mp4')))
print(len(videos), 'videos')
if videos:
    data = base64.b64encode(open(videos[0], 'rb').read()).decode()
    display(HTML(f'<video width=480 controls><source src="data:video/mp4;base64,{data}" type="video/mp4"></video>'))

## 9. Download everything

Grab the CSVs and figures so you can commit them and drop the plots into the write-up.

In [ ]:
!cd {OUT_DIR} && zip -qr /content/ppo_results.zip *.csv figures
from google.colab import files
files.download('/content/ppo_results.zip')